In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('../db_defense_systems/DefenseFinder/df-RefSeq.csv')

In [5]:
# Assuming your DataFrame is named df
# First, split each comma-separated string into lists
df_expanded = df.assign(
    name_of_profiles_in_sys=df['name_of_profiles_in_sys'].str.split(','),
    accession_in_sys=df['accession_in_sys'].str.split(','),
    protein_in_syst=df['protein_in_syst'].str.split(',')
)

# Then explode all three columns simultaneously to maintain alignment
df_expanded = df_expanded.explode(['name_of_profiles_in_sys', 'accession_in_sys', 'protein_in_syst'])

# Reset index if needed
df_expanded = df_expanded.reset_index(drop=True)

In [6]:
from Bio import Entrez
from Bio import SeqIO
import time
from tqdm import tqdm 

In [9]:
Entrez.email = "freddie.martin@cpr.ku.dk"  # Replace with your email
Entrez.api_key = "fd2d0b33ef50e336785a7617e5f285310107"  # Get from https://www.ncbi.nlm.nih.gov/account/

def fetch_protein_sequences_batch(accessions, batch_size=200, wait=0.1,max_retries=3):
    """Fetch multiple protein sequences in batches."""
    accession_to_sequence = {}
    
    for i in tqdm(range(0, len(accessions), batch_size)):
        batch = accessions[i:i + batch_size]
        
        for attempt in range(max_retries):
            try:
                handle = Entrez.efetch(
                    db="protein", 
                    id=",".join(batch),
                    rettype="fasta", 
                    retmode="text"
                )
                records = SeqIO.parse(handle, "fasta")
                
                for record in records:
                    acc = record.id.split("|")[3] if "|" in record.id else record.id
                    accession_to_sequence[acc] = str(record.seq)
                
                handle.close()
                break  # Success, exit retry loop
                
            except Exception as e:
                if attempt == max_retries - 1:
                    print(f"Failed batch after {max_retries} attempts: {e}")
                else:
                    time.sleep(1)  # Wait before retry
        
        time.sleep(wait)  # or 0.1 with API key
            
    return accession_to_sequence



In [10]:
unique_accessions = df_expanded['accession_in_sys'].unique().tolist()
accession_to_sequence = fetch_protein_sequences_batch(unique_accessions)
df_expanded['protein_sequence'] = df_expanded['accession_in_sys'].map(accession_to_sequence)

 48%|██████████████████                    | 473/992 [14:51<18:39,  2.16s/it]

Failed batch after 3 attempts: sequence item 38: expected str instance, float found


100%|██████████████████████████████████████| 992/992 [31:05<00:00,  1.88s/it]


In [6]:
df_expanded.shape

(380023, 20)

In [ ]:
# Use it
unique_accessions = df_expanded['accession_in_sys'].unique().tolist()
accession_to_sequence = fetch_protein_sequences_batch(unique_accessions)
df_expanded['protein_sequence'] = df_expanded['accession_in_sys'].map(accession_to_sequence)

unique_accessions = df_expanded['accession_in_sys'].unique()
accession_to_sequence = {}

for accession in tqdm(unique_accessions, desc="Fetching protein sequences"):
    sequence = fetch_protein_sequence(accession)
    if sequence:
        accession_to_sequence[accession] = sequence
    time.sleep(0.34)

# Add sequences to the DataFrame
df_expanded['protein_sequence'] = df_expanded['accession_in_sys'].map(accession_to_sequence)

In [3]:
df_expanded = pd.read_csv('df-sequences.csv')

In [4]:
df_expanded[['sys_id', 'protein_sequence']]

,sys_id,protein_sequence
0,GCF_023612275_NZ_CP063766_Abi2_1,MKKYGSKQEYVKRPKSIINNRFPSKLKKKIRPNQYFYSRVRCQIIP...
1,GCF_900660665_NZ_LR215037_Abi2_1,MNKNNVKSLLSFEEQLNFLQSKYINWNNQEKFRFKIYLINYGYSHM...
2,GCF_900660665_NZ_LR215037_Abi2_2,MNKKIKTILTTDEQINLLEEWGLKWEDDYSRNILKIYLKNYGYKHI...
3,GCF_008365255_NZ_CP043501_Abi2_1,MVKGKTTNGLMRHLREKHGISINGSKHKKELLNMGYYHGYKGYRFI...
4,GCF_009296125_NZ_CP017197_Abi2_1,MTIYDTEQIKSFLGLRQIKSNDDFKAVETMSTDTAIKFKKYVEYLG...
...,...,...
380018,GCF_012029695_NZ_CP061389_Zorya_TypeI_1,MSWLNSFLVTLTSVEPHNVPFTVISIVSLAVACFLFFYLFRAIKIV...
380019,GCF_002356355_NZ_AP018052_Zorya_TypeI_2,MSTKHLAELLDRLANAIVTLEGVALICILIALVALYNYLRFRARIA...
380020,GCF_002356355_NZ_AP018052_Zorya_TypeI_2,MKRPAEHTTFEDDGAGYLISVSDMMAGLLFVFMITLMAFVINFQQA...
380021,GCF_002356355_NZ_AP018052_Zorya_TypeI_2,MAGLKESLSSRVVTSPHSESWPAPALETAIREARWKLARVEPRDPP...


In [5]:
len(df_expanded['subtype'].unique())

265

## Using data sampling

In [39]:
df_sampled = (
    df_expanded
    .groupby('subtype', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), 100), random_state=42))
)

/var/folders/k0/7lzlp98x6c188pkk0rzbn2d40000gn/T/ipykernel_95615/2102740797.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 100), random_state=42))


In [48]:
df_sampled.head()

,Unnamed: 0,sys_id,Assembly,replicon,type,subtype,sys_beg,sys_end,protein_in_syst,genes_count,...,accession_in_sys,Superkingdom,phylum,class,order,family,genus,species,protein_sequence,cluster_rep
1241,1241,GCF_900635335_NZ_LR134093_Abi2_1,GCF_900635335.1,NZ_LR134093,Abi2,Abi2,GCF_900635335.1_NZ_LR134093_00335,GCF_900635335.1_NZ_LR134093_00335,GCF_900635335.1_NZ_LR134093_00335,1.0,...,WP_001080636.1,Bacteria,Bacillota,Bacilli,Bacillales,Staphylococcaceae,Staphylococcus,Staphylococcus aureus,MNRKTPKTTDGLMRHIRDNKGIQINGSTEKNQLRNIGYFHGFKGYN...,NaN
762,762,GCF_003031285_NZ_CP023561_Abi2_1,GCF_003031285.1,NZ_CP023561,Abi2,Abi2,GCF_003031285.1_NZ_CP023561_00636,GCF_003031285.1_NZ_CP023561_00636,GCF_003031285.1_NZ_CP023561_00636,1.0,...,WP_001822249.1,Bacteria,Bacillota,Bacilli,Bacillales,Staphylococcaceae,Staphylococcus,Staphylococcus aureus,MLNFDEQIAKLKQMNIFFNIIDTEKANEILRKNNYFFKLAYFRKNF...,NaN
1220,1220,GCF_900607295_NZ_LR027869_Abi2_1,GCF_900607295.1,NZ_LR027869,Abi2,Abi2,GCF_900607295.1_NZ_LR027869_00376,GCF_900607295.1_NZ_LR027869_00376,GCF_900607295.1_NZ_LR027869_00376,1.0,...,WP_001822249.1,Bacteria,Bacillota,Bacilli,Bacillales,Staphylococcaceae,Staphylococcus,Staphylococcus aureus,MLNFDEQIAKLKQMNIFFNIIDTEKANEILRKNNYFFKLAYFRKNF...,NaN
771,771,GCF_003595425_NZ_CP014371_Abi2_1,GCF_003595425.1,NZ_CP014371,Abi2,Abi2,GCF_003595425.1_NZ_CP014371_00386,GCF_003595425.1_NZ_CP014371_00386,GCF_003595425.1_NZ_CP014371_00386,1.0,...,WP_001822249.1,Bacteria,Bacillota,Bacilli,Bacillales,Staphylococcaceae,Staphylococcus,Staphylococcus aureus,MLNFDEQIAKLKQMNIFFNIIDTEKANEILRKNNYFFKLAYFRKNF...,NaN
668,668,GCF_003073715_NZ_CP029087_Abi2_1,GCF_003073715.1,NZ_CP029087,Abi2,Abi2,GCF_003073715.1_NZ_CP029087_00367,GCF_003073715.1_NZ_CP029087_00367,GCF_003073715.1_NZ_CP029087_00367,1.0,...,WP_108711085.1,Bacteria,Bacillota,Bacilli,Bacillales,Staphylococcaceae,Staphylococcus,Staphylococcus aureus,MLNFDEQIAKLKQMNIFFNIIDTEKANEILRKNNYFFKLAYFRKNF...,NaN


In [46]:

with open("proteins_for_deeptmhmm_1.fasta", "w") as f:
    for idx, row in df_sampled.iloc[0:int(len(df_sampled)/2)].iterrows():
        header = f">{idx}|{row['sys_id']}"
        sequence = row['protein_sequence']
        f.write(f"{header}\n")
        f.write(f"{sequence}\n")


In [58]:
import re

def read_tmr_counts(gff_path):
    """
    Returns a dict:
      { idx (int) : number_of_predicted_TMRs (int) }
    """
    tmr = {}

    current_idx = None

    with open(gff_path) as f:
        for line in f:
            line = line.strip()

            # Match TMR count line
            # Match header line with ID
            m_tmr = re.match(r"#\s+(\d+)\|.*Number of predicted TMRs:\s+(\d+)", line)
            if m_tmr is not None:
                current_idx = int(m_tmr.group(1))
                tmr[current_idx] = int(m_tmr.group(2))
                continue

    return tmr


In [60]:
tmr_counts = {}
tmr_counts.update(read_tmr_counts("./results_sampled_1/TMRs.gff3"))
tmr_counts.update(read_tmr_counts("./results_sampled_2/TMRs.gff3"))

In [62]:
df_sampled["predicted_TMRs"] = df_sampled.index.map(tmr_counts)


In [63]:
df_sampled

,Unnamed: 0,sys_id,Assembly,replicon,type,subtype,sys_beg,sys_end,protein_in_syst,genes_count,...,Superkingdom,phylum,class,order,family,genus,species,protein_sequence,cluster_rep,predicted_TMRs
1241,1241,GCF_900635335_NZ_LR134093_Abi2_1,GCF_900635335.1,NZ_LR134093,Abi2,Abi2,GCF_900635335.1_NZ_LR134093_00335,GCF_900635335.1_NZ_LR134093_00335,GCF_900635335.1_NZ_LR134093_00335,1.0,...,Bacteria,Bacillota,Bacilli,Bacillales,Staphylococcaceae,Staphylococcus,Staphylococcus aureus,MNRKTPKTTDGLMRHIRDNKGIQINGSTEKNQLRNIGYFHGFKGYN...,NaN,0
762,762,GCF_003031285_NZ_CP023561_Abi2_1,GCF_003031285.1,NZ_CP023561,Abi2,Abi2,GCF_003031285.1_NZ_CP023561_00636,GCF_003031285.1_NZ_CP023561_00636,GCF_003031285.1_NZ_CP023561_00636,1.0,...,Bacteria,Bacillota,Bacilli,Bacillales,Staphylococcaceae,Staphylococcus,Staphylococcus aureus,MLNFDEQIAKLKQMNIFFNIIDTEKANEILRKNNYFFKLAYFRKNF...,NaN,0
1220,1220,GCF_900607295_NZ_LR027869_Abi2_1,GCF_900607295.1,NZ_LR027869,Abi2,Abi2,GCF_900607295.1_NZ_LR027869_00376,GCF_900607295.1_NZ_LR027869_00376,GCF_900607295.1_NZ_LR027869_00376,1.0,...,Bacteria,Bacillota,Bacilli,Bacillales,Staphylococcaceae,Staphylococcus,Staphylococcus aureus,MLNFDEQIAKLKQMNIFFNIIDTEKANEILRKNNYFFKLAYFRKNF...,NaN,0
771,771,GCF_003595425_NZ_CP014371_Abi2_1,GCF_003595425.1,NZ_CP014371,Abi2,Abi2,GCF_003595425.1_NZ_CP014371_00386,GCF_003595425.1_NZ_CP014371_00386,GCF_003595425.1_NZ_CP014371_00386,1.0,...,Bacteria,Bacillota,Bacilli,Bacillales,Staphylococcaceae,Staphylococcus,Staphylococcus aureus,MLNFDEQIAKLKQMNIFFNIIDTEKANEILRKNNYFFKLAYFRKNF...,NaN,0
668,668,GCF_003073715_NZ_CP029087_Abi2_1,GCF_003073715.1,NZ_CP029087,Abi2,Abi2,GCF_003073715.1_NZ_CP029087_00367,GCF_003073715.1_NZ_CP029087_00367,GCF_003073715.1_NZ_CP029087_00367,1.0,...,Bacteria,Bacillota,Bacilli,Bacillales,Staphylococcaceae,Staphylococcus,Staphylococcus aureus,MLNFDEQIAKLKQMNIFFNIIDTEKANEILRKNNYFFKLAYFRKNF...,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224730,224730,GCF_009834325_NZ_CP034754_radar_II_1,GCF_009834325.1,NZ_CP034754,RADAR,radar_II,GCF_009834325.1_NZ_CP034754_03483,GCF_009834325.1_NZ_CP034754_03485,GCF_009834325.1_NZ_CP034754_03485,3.0,...,Bacteria,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Enterobacteriaceae,Enterobacter,Enterobacter hormaechei,MSTHRFIPLDVSESAVVKDSRSLLPLDVYKKIAQFIRKALDNLPES...,NaN,0
224958,224958,GCF_002201815_NZ_CP021896_radar_II_1,GCF_002201815.1,NZ_CP021896,RADAR,radar_II,GCF_002201815.1_NZ_CP021896_04096,GCF_002201815.1_NZ_CP021896_04098,GCF_002201815.1_NZ_CP021896_04097,3.0,...,Bacteria,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Enterobacteriaceae,Enterobacter,Enterobacter cloacae complex sp.,MDKLCDGDLFHTGRLPWLTKIFEQLMVRNGDLICYRETEVQAYARL...,NaN,0
224944,224944,GCF_021725435_NZ_CP091319_radar_II_1,GCF_021725435.1,NZ_CP091319,RADAR,radar_II,GCF_021725435.1_NZ_CP091319_03315,GCF_021725435.1_NZ_CP091319_03317,GCF_021725435.1_NZ_CP091319_03315,3.0,...,Bacteria,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Enterobacteriaceae,Enterobacter,Enterobacter sp. JH25,MHKEPVVIASFPQLIHWFDDEAKRLLAKSEQVRMLSYLFRRLSLYF...,NaN,4
224949,224949,GCF_003031445_NZ_CP020525_radar_II_4,GCF_003031445.1,NZ_CP020525,RADAR,radar_II,GCF_003031445.1_NZ_CP020525_03391,GCF_003031445.1_NZ_CP020525_03393,GCF_003031445.1_NZ_CP020525_03393,3.0,...,Bacteria,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Enterobacteriaceae,Enterobacter,Enterobacter cloacae,MSTHRFIPLDVSESAVVKDSRSLLPLDVYKKIAQFIRKALDNLPES...,NaN,0


In [64]:
df_sampled["predicted_TMRs"].value_counts(dropna=False)


predicted_TMRs
0    21064
2      517
1      486
4      234
3      177
6        2
Name: count, dtype: int64

In [65]:
df_sampled["TMR_present"] = df_sampled["predicted_TMRs"].gt(0)


In [66]:
df_sampled['TMR_present'].value_counts()

TMR_present
False    21064
True      1416
Name: count, dtype: int64

In [67]:
df_sampled.to_excel('df_DefenseFinder_DeepTMHMM_sampled_results.xlsx')

## Clustering sequences

In [ ]:
!mmseqs easy-cluster proteins_for_deeptmhmm.fasta clusters tmp \
  --min-seq-id 0.85 \
  -c 0.70


In [68]:
clusters = pd.read_csv(
    "./clustered_85/clusters_cluster.tsv",
    sep="\t",
    names=["rep_idx", "member_idx"]
)


In [74]:
df_expanded

,Unnamed: 0,sys_id,Assembly,replicon,type,subtype,sys_beg,sys_end,protein_in_syst,genes_count,...,accession_in_sys,Superkingdom,phylum,class,order,family,genus,species,protein_sequence,cluster_rep
0,0,GCF_023612275_NZ_CP063766_Abi2_1,GCF_023612275.1,NZ_CP063766,Abi2,Abi2,GCF_023612275.1_NZ_CP063766_01013,GCF_023612275.1_NZ_CP063766_01013,GCF_023612275.1_NZ_CP063766_01013,1.0,...,WP_010728290.1,Bacteria,Bacillota,Bacilli,Lactobacillales,Enterococcaceae,Enterococcus,Enterococcus lactis,MKKYGSKQEYVKRPKSIINNRFPSKLKKKIRPNQYFYSRVRCQIIP...,NaN
1,1,GCF_900660665_NZ_LR215037_Abi2_1,GCF_900660665.1,NZ_LR215037,Abi2,Abi2,GCF_900660665.1_NZ_LR215037_00128,GCF_900660665.1_NZ_LR215037_00128,GCF_900660665.1_NZ_LR215037_00128,1.0,...,WP_129646131.1,Bacteria,Mycoplasmatota,Mollicutes,Mycoplasmatales,Mycoplasmataceae,Mycoplasmopsis,Mycoplasmopsis maculosa,MNKNNVKSLLSFEEQLNFLQSKYINWNNQEKFRFKIYLINYGYSHM...,NaN
2,2,GCF_900660665_NZ_LR215037_Abi2_2,GCF_900660665.1,NZ_LR215037,Abi2,Abi2,GCF_900660665.1_NZ_LR215037_00677,GCF_900660665.1_NZ_LR215037_00677,GCF_900660665.1_NZ_LR215037_00677,1.0,...,WP_129647121.1,Bacteria,Mycoplasmatota,Mollicutes,Mycoplasmatales,Mycoplasmataceae,Mycoplasmopsis,Mycoplasmopsis maculosa,MNKKIKTILTTDEQINLLEEWGLKWEDDYSRNILKIYLKNYGYKHI...,NaN
3,3,GCF_008365255_NZ_CP043501_Abi2_1,GCF_008365255.1,NZ_CP043501,Abi2,Abi2,GCF_008365255.1_NZ_CP043501_01947,GCF_008365255.1_NZ_CP043501_01947,GCF_008365255.1_NZ_CP043501_01947,1.0,...,WP_149208562.1,Bacteria,Bacillota,Bacilli,Bacillales,Bacillaceae,Bacillus,Bacillus paralicheniformis,MVKGKTTNGLMRHLREKHGISINGSKHKKELLNMGYYHGYKGYRFI...,NaN
4,4,GCF_009296125_NZ_CP017197_Abi2_1,GCF_009296125.1,NZ_CP017197,Abi2,Abi2,GCF_009296125.1_NZ_CP017197_01353,GCF_009296125.1_NZ_CP017197_01353,GCF_009296125.1_NZ_CP017197_01353,1.0,...,WP_097001356.1,Bacteria,Bacillota,Bacilli,Lactobacillales,Lactobacillaceae,Leuconostoc,Leuconostoc gasicomitatum,MTIYDTEQIKSFLGLRQIKSNDDFKAVETMSTDTAIKFKKYVEYLG...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
380018,380018,GCF_012029695_NZ_CP061389_Zorya_TypeI_1,GCF_012029695.2,NZ_CP061389,Zorya,Zorya_TypeI,GCF_012029695.2_NZ_CP061389_04302,GCF_012029695.2_NZ_CP061389_04305,GCF_012029695_NZ_CP061389_04305,4.0,...,WP_004178700.1,Bacteria,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Enterobacteriaceae,Klebsiella,Klebsiella pneumoniae,MSWLNSFLVTLTSVEPHNVPFTVISIVSLAVACFLFFYLFRAIKIV...,NaN
380019,380019,GCF_002356355_NZ_AP018052_Zorya_TypeI_2,GCF_002356355.1,NZ_AP018052,Zorya,Zorya_TypeI,GCF_002356355.1_NZ_AP018052_01338,GCF_002356355.1_NZ_AP018052_01341,GCF_002356355_NZ_AP018052_01338,4.0,...,WP_096365918.1,Bacteria,Pseudomonadota,Gammaproteobacteria,Chromatiales,NaN,Thiohalobacter,Thiohalobacter thiocyanaticus,MSTKHLAELLDRLANAIVTLEGVALICILIALVALYNYLRFRARIA...,NaN
380020,380020,GCF_002356355_NZ_AP018052_Zorya_TypeI_2,GCF_002356355.1,NZ_AP018052,Zorya,Zorya_TypeI,GCF_002356355.1_NZ_AP018052_01338,GCF_002356355.1_NZ_AP018052_01341,GCF_002356355_NZ_AP018052_01339,4.0,...,WP_197703019.1,Bacteria,Pseudomonadota,Gammaproteobacteria,Chromatiales,NaN,Thiohalobacter,Thiohalobacter thiocyanaticus,MKRPAEHTTFEDDGAGYLISVSDMMAGLLFVFMITLMAFVINFQQA...,NaN
380021,380021,GCF_002356355_NZ_AP018052_Zorya_TypeI_2,GCF_002356355.1,NZ_AP018052,Zorya,Zorya_TypeI,GCF_002356355.1_NZ_AP018052_01338,GCF_002356355.1_NZ_AP018052_01341,GCF_002356355_NZ_AP018052_01340,4.0,...,WP_157745444.1,Bacteria,Pseudomonadota,Gammaproteobacteria,Chromatiales,NaN,Thiohalobacter,Thiohalobacter thiocyanaticus,MAGLKESLSSRVVTSPHSESWPAPALETAIREARWKLARVEPRDPP...,NaN


In [69]:
df_expanded["cluster_rep"] = df_expanded.index.map(
    clusters.set_index("member_idx")["rep_idx"]
)


In [11]:
df_reps = df_expanded[df_expanded.index == df_expanded["cluster_rep"]]


In [72]:
from pathlib import Path

input_fasta = "./clustered_85/clusters_rep_seq.fasta"
num_batches = 8
output_prefix = "proteins_for_deeptmhmm_batch"

# Read FASTA records
records = []
with open(input_fasta) as f:
    record = []
    for line in f:
        if line.startswith(">") and record:
            records.append("".join(record))
            record = [line]
        else:
            record.append(line)
    if record:
        records.append("".join(record))

# Split records into batches
batch_size = (len(records) + num_batches - 1) // num_batches

for i in range(num_batches):
    batch_records = records[i * batch_size:(i + 1) * batch_size]
    if not batch_records:
        break

    out_file = f"{output_prefix}_{i+1}.fasta"
    with open(out_file, "w") as f:
        f.writelines(batch_records)

    print(f"Wrote {len(batch_records)} records to {out_file}")


Wrote 15390 records to proteins_for_deeptmhmm_batch_1.fasta
Wrote 15390 records to proteins_for_deeptmhmm_batch_2.fasta
Wrote 15390 records to proteins_for_deeptmhmm_batch_3.fasta
Wrote 15390 records to proteins_for_deeptmhmm_batch_4.fasta
Wrote 15390 records to proteins_for_deeptmhmm_batch_5.fasta
Wrote 15390 records to proteins_for_deeptmhmm_batch_6.fasta
Wrote 15390 records to proteins_for_deeptmhmm_batch_7.fasta
Wrote 15383 records to proteins_for_deeptmhmm_batch_8.fasta


In [12]:
with open("proteins_clustered_90.fasta", "w") as f:
    for idx, seq in zip(df_reps.index, df_reps["protein_sequence"]):
        f.write(f">{idx}\n{seq}\n")


In [75]:
import re
import pandas as pd
from pathlib import Path

gff_paths = ['../DeepTMHMM-Academic-License-v1.0/results_batch_1/TMRs.gff3',
             '../DeepTMHMM-Academic-License-v1.0/results_batch_2/TMRs.gff3',
             '../DeepTMHMM-Academic-License-v1.0/results_batch_3/TMRs.gff3',
             '../DeepTMHMM-Academic-License-v1.0/results_batch_4/TMRs.gff3',
             '../DeepTMHMM-Academic-License-v1.0/results_batch_5/TMRs.gff3',
             '../DeepTMHMM-Academic-License-v1.0/results_batch_6/TMRs.gff3',
             '../DeepTMHMM-Academic-License-v1.0/results_batch_7/TMRs.gff3',
             '../DeepTMHMM-Academic-License-v1.0/results_batch_8/TMRs.gff3',
            ]  # directory with all .gff3 files

records = []

tmr_re = re.compile(r"#\s+(?P<seqid>\S+)\s+Number of predicted TMRs:\s+(?P<n>\d+)")

for gff in gff_paths:
    with open(gff) as f:
        for line in f:
            m = tmr_re.match(line)
            if m:
                records.append({
                    "cluster_rep": m.group("seqid"),
                    "n_tmr": int(m.group("n"))
                })

deeptmhmm_df = pd.DataFrame(records).drop_duplicates("cluster_rep")


In [ ]:
three_line_paths = [
    "../DeepTMHMM-Academic-License-v1.0/results_batch_1/predicted_topologies.3line",
    "../DeepTMHMM-Academic-License-v1.0/results_batch_2/predicted_topologies.3line",
    "../DeepTMHMM-Academic-License-v1.0/results_batch_3/predicted_topologies.3line",
    "../DeepTMHMM-Academic-License-v1.0/results_batch_4/predicted_topologies.3line",
    "../DeepTMHMM-Academic-License-v1.0/results_batch_5/predicted_topologies.3line",
    "../DeepTMHMM-Academic-License-v1.0/results_batch_6/predicted_topologies.3line",
    "../DeepTMHMM-Academic-License-v1.0/results_batch_7/predicted_topologies.3line",
    "../DeepTMHMM-Academic-License-v1.0/results_batch_8/predicted_topologies.3line",
]
from pathlib import Path

three_line_files = []

import re
import pandas as pd
from pathlib import Path

gff_paths = ['../DeepTMHMM-Academic-License-v1.0/results_batch_1/TMRs.gff3',
             '../DeepTMHMM-Academic-License-v1.0/results_batch_2/TMRs.gff3',
             '../DeepTMHMM-Academic-License-v1.0/results_batch_3/TMRs.gff3',
             '../DeepTMHMM-Academic-License-v1.0/results_batch_4/TMRs.gff3',
             '../DeepTMHMM-Academic-License-v1.0/results_batch_5/TMRs.gff3',
             '../DeepTMHMM-Academic-License-v1.0/results_batch_6/TMRs.gff3',
             '../DeepTMHMM-Academic-License-v1.0/results_batch_7/TMRs.gff3',
             '../DeepTMHMM-Academic-License-v1.0/results_batch_8/TMRs.gff3',
            ]  # directory with all .gff3 files

records = []

tmr_re = re.compile(r"#\s+(?P<seqid>\S+)\s+Number of predicted TMRs:\s+(?P<n>\d+)")

for gff in gff_paths:
    with open(gff) as f:
        for line in f:
            m = tmr_re.match(line)
            if m:
                records.append({
                    "cluster_rep": m.group("seqid"),
                    "n_tmr": int(m.group("n"))
                })

deeptmhmm_df = pd.DataFrame(records).drop_duplicates("cluster_rep")


for p in three_line_paths:
    p = Path(p)
    if p.is_dir():
        three_line_files.extend(p.rglob("*.3line"))
    elif p.is_file() and p.suffix == ".3line":
        three_line_files.append(p)
import pandas as pd

records = []

for file in three_line_files:
    with open(file) as f:
        while True:
            header = f.readline().rstrip()
            if not header:
                break

            seq = f.readline().rstrip()
            topo = f.readline().rstrip()

            # >346261|GCF_... | GLOB
            header_core = header[1:].split("|", 1)
            idx = int(header_core[0])
            rest = header_core[1]

            protein_id = rest.split(" | ")[0]

            n_tmr = topo.count("M")

            records.append({
                "cluster_rep_idx": idx,
                "protein_id": protein_id,
                "protein_sequence": seq,
                "n_tmr": n_tmr,
                "is_membrane": n_tmr > 0,
                "source_file": str(file)   # optional, very useful for debugging
            })

deeptmhmm_df = pd.DataFrame(records)

# same sequence should never have conflicting predictions
assert deeptmhmm_df.groupby("protein_sequence")["n_tmr"].nunique().max() == 1

# reduce to unique predictions
deeptmhmm_df = deeptmhmm_df.drop_duplicates(
    subset=["cluster_rep_idx", "protein_sequence"]
)


In [111]:
three_line_paths = [
    "../DeepTMHMM-Academic-License-v1.0/results_batch_1/predicted_topologies.3line",
    "../DeepTMHMM-Academic-License-v1.0/results_batch_2/predicted_topologies.3line",
    "../DeepTMHMM-Academic-License-v1.0/results_batch_3/predicted_topologies.3line",
    "../DeepTMHMM-Academic-License-v1.0/results_batch_4/predicted_topologies.3line",
    "../DeepTMHMM-Academic-License-v1.0/results_batch_5/predicted_topologies.3line",
    "../DeepTMHMM-Academic-License-v1.0/results_batch_6/predicted_topologies.3line",
    "../DeepTMHMM-Academic-License-v1.0/results_batch_7/predicted_topologies.3line",
    "../DeepTMHMM-Academic-License-v1.0/results_batch_8/predicted_topologies.3line",
]
from pathlib import Path

three_line_files = []

for p in three_line_paths:
    p = Path(p)
    if p.is_dir():
        three_line_files.extend(p.rglob("*.3line"))
    elif p.is_file() and p.suffix == ".3line":
        three_line_files.append(p)
import pandas as pd

records = []

for file in three_line_files:
    with open(file) as f:
        while True:
            header = f.readline().rstrip()
            if not header:
                break

            seq = f.readline().rstrip()
            topo = f.readline().rstrip()

            # >346261|GCF_... | GLOB
            header_core = header[1:].split("|", 1)
            idx = int(header_core[0])
            rest = header_core[1]

            protein_id = rest.split(" | ")[0]

            n_tmr = topo.count("M")

            records.append({
                "cluster_rep_idx": idx,
                "protein_id": protein_id,
                "protein_sequence": seq,
                "n_tmr": n_tmr,
                "is_membrane": n_tmr > 0,
                "source_file": str(file)   # optional, very useful for debugging
            })

deeptmhmm_df = pd.DataFrame(records)

# same sequence should never have conflicting predictions
assert deeptmhmm_df.groupby("protein_sequence")["n_tmr"].nunique().max() == 1

# reduce to unique predictions
deeptmhmm_df = deeptmhmm_df.drop_duplicates(
    subset=["cluster_rep_idx", "protein_sequence","protein_id"]
)


In [126]:
df_expanded

,Unnamed: 0,sys_id,Assembly,replicon,type,subtype,sys_beg,sys_end,protein_in_syst,genes_count,...,phylum,class,order,family,genus,species,protein_sequence,cluster_rep_x,cluster_rep_y,member
0,0,GCF_023612275_NZ_CP063766_Abi2_1,GCF_023612275.1,NZ_CP063766,Abi2,Abi2,GCF_023612275.1_NZ_CP063766_01013,GCF_023612275.1_NZ_CP063766_01013,GCF_023612275.1_NZ_CP063766_01013,1.0,...,Bacillota,Bacilli,Lactobacillales,Enterococcaceae,Enterococcus,Enterococcus lactis,MKKYGSKQEYVKRPKSIINNRFPSKLKKKIRPNQYFYSRVRCQIIP...,NaN,NaN,NaN
1,1,GCF_900660665_NZ_LR215037_Abi2_1,GCF_900660665.1,NZ_LR215037,Abi2,Abi2,GCF_900660665.1_NZ_LR215037_00128,GCF_900660665.1_NZ_LR215037_00128,GCF_900660665.1_NZ_LR215037_00128,1.0,...,Mycoplasmatota,Mollicutes,Mycoplasmatales,Mycoplasmataceae,Mycoplasmopsis,Mycoplasmopsis maculosa,MNKNNVKSLLSFEEQLNFLQSKYINWNNQEKFRFKIYLINYGYSHM...,NaN,NaN,NaN
2,2,GCF_900660665_NZ_LR215037_Abi2_2,GCF_900660665.1,NZ_LR215037,Abi2,Abi2,GCF_900660665.1_NZ_LR215037_00677,GCF_900660665.1_NZ_LR215037_00677,GCF_900660665.1_NZ_LR215037_00677,1.0,...,Mycoplasmatota,Mollicutes,Mycoplasmatales,Mycoplasmataceae,Mycoplasmopsis,Mycoplasmopsis maculosa,MNKKIKTILTTDEQINLLEEWGLKWEDDYSRNILKIYLKNYGYKHI...,NaN,NaN,NaN
3,3,GCF_008365255_NZ_CP043501_Abi2_1,GCF_008365255.1,NZ_CP043501,Abi2,Abi2,GCF_008365255.1_NZ_CP043501_01947,GCF_008365255.1_NZ_CP043501_01947,GCF_008365255.1_NZ_CP043501_01947,1.0,...,Bacillota,Bacilli,Bacillales,Bacillaceae,Bacillus,Bacillus paralicheniformis,MVKGKTTNGLMRHLREKHGISINGSKHKKELLNMGYYHGYKGYRFI...,NaN,NaN,NaN
4,4,GCF_009296125_NZ_CP017197_Abi2_1,GCF_009296125.1,NZ_CP017197,Abi2,Abi2,GCF_009296125.1_NZ_CP017197_01353,GCF_009296125.1_NZ_CP017197_01353,GCF_009296125.1_NZ_CP017197_01353,1.0,...,Bacillota,Bacilli,Lactobacillales,Lactobacillaceae,Leuconostoc,Leuconostoc gasicomitatum,MTIYDTEQIKSFLGLRQIKSNDDFKAVETMSTDTAIKFKKYVEYLG...,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
380018,380018,GCF_012029695_NZ_CP061389_Zorya_TypeI_1,GCF_012029695.2,NZ_CP061389,Zorya,Zorya_TypeI,GCF_012029695.2_NZ_CP061389_04302,GCF_012029695.2_NZ_CP061389_04305,GCF_012029695_NZ_CP061389_04305,4.0,...,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Enterobacteriaceae,Klebsiella,Klebsiella pneumoniae,MSWLNSFLVTLTSVEPHNVPFTVISIVSLAVACFLFFYLFRAIKIV...,NaN,NaN,NaN
380019,380019,GCF_002356355_NZ_AP018052_Zorya_TypeI_2,GCF_002356355.1,NZ_AP018052,Zorya,Zorya_TypeI,GCF_002356355.1_NZ_AP018052_01338,GCF_002356355.1_NZ_AP018052_01341,GCF_002356355_NZ_AP018052_01338,4.0,...,Pseudomonadota,Gammaproteobacteria,Chromatiales,NaN,Thiohalobacter,Thiohalobacter thiocyanaticus,MSTKHLAELLDRLANAIVTLEGVALICILIALVALYNYLRFRARIA...,NaN,NaN,NaN
380020,380020,GCF_002356355_NZ_AP018052_Zorya_TypeI_2,GCF_002356355.1,NZ_AP018052,Zorya,Zorya_TypeI,GCF_002356355.1_NZ_AP018052_01338,GCF_002356355.1_NZ_AP018052_01341,GCF_002356355_NZ_AP018052_01339,4.0,...,Pseudomonadota,Gammaproteobacteria,Chromatiales,NaN,Thiohalobacter,Thiohalobacter thiocyanaticus,MKRPAEHTTFEDDGAGYLISVSDMMAGLLFVFMITLMAFVINFQQA...,NaN,NaN,NaN
380021,380021,GCF_002356355_NZ_AP018052_Zorya_TypeI_2,GCF_002356355.1,NZ_AP018052,Zorya,Zorya_TypeI,GCF_002356355.1_NZ_AP018052_01338,GCF_002356355.1_NZ_AP018052_01341,GCF_002356355_NZ_AP018052_01340,4.0,...,Pseudomonadota,Gammaproteobacteria,Chromatiales,NaN,Thiohalobacter,Thiohalobacter thiocyanaticus,MAGLKESLSSRVVTSPHSESWPAPALETAIREARWKLARVEPRDPP...,NaN,NaN,NaN


In [118]:
deeptmhmm_df.head()

,cluster_rep_idx,protein_id,protein_sequence,n_tmr,is_membrane,source_file
0,0,GCF_023612275_NZ_CP063766_Abi2_1,MKKYGSKQEYVKRPKSIINNRFPSKLKKKIRPNQYFYSRVRCQIIP...,0,False,../DeepTMHMM-Academic-License-v1.0/results_bat...
1,1,GCF_900660665_NZ_LR215037_Abi2_1,MNKNNVKSLLSFEEQLNFLQSKYINWNNQEKFRFKIYLINYGYSHM...,0,False,../DeepTMHMM-Academic-License-v1.0/results_bat...
2,2,GCF_900660665_NZ_LR215037_Abi2_2,MNKKIKTILTTDEQINLLEEWGLKWEDDYSRNILKIYLKNYGYKHI...,0,False,../DeepTMHMM-Academic-License-v1.0/results_bat...
3,3,GCF_008365255_NZ_CP043501_Abi2_1,MVKGKTTNGLMRHLREKHGISINGSKHKKELLNMGYYHGYKGYRFI...,0,False,../DeepTMHMM-Academic-License-v1.0/results_bat...
4,4,GCF_009296125_NZ_CP017197_Abi2_1,MTIYDTEQIKSFLGLRQIKSNDDFKAVETMSTDTAIKFKKYVEYLG...,0,False,../DeepTMHMM-Academic-License-v1.0/results_bat...


In [119]:
clusters = pd.read_csv(
    "./clustered_85/clusters_cluster.tsv",
    sep="\t",
    header=None,
    names=["cluster_rep", "member"]
)

# remove numeric prefix before '|'
clusters[["cluster_idx","cluster_rep",]] = clusters["cluster_rep"].str.split("|", n=1, expand=True)
clusters[["member_idx","member",]]  = clusters["member"].str.split("|", n=1, expand=True)



In [120]:
clusters = clusters.reset_index()

In [123]:
clusters

,index,cluster_rep,member,cluster_idx,member_idx
0,0,GCF_023612275_NZ_CP063766_Abi2_1,GCF_023612275_NZ_CP063766_Abi2_1,0,0
1,1,GCF_023612275_NZ_CP063766_Abi2_1,GCF_001587115_NZ_CP011828_Abi2_1,0,24
2,2,GCF_023612275_NZ_CP063766_Abi2_1,GCF_015377765_NZ_CP045602_Abi2_1,0,32
3,3,GCF_023612275_NZ_CP063766_Abi2_1,GCF_002216705_NZ_CP022340_Abi2_1,0,243
4,4,GCF_023612275_NZ_CP063766_Abi2_1,GCF_003667965_NZ_CP033041_Abi2_1,0,321
...,...,...,...,...,...
380018,380018,GCF_009662315_NZ_CP045739_RM_Type_IV_5,GCF_900243355_NZ_LT969520_RM_Type_IV_2,279114,279285
380019,380019,GCF_009662315_NZ_CP045739_RM_Type_IV_5,GCF_023093935_NZ_CP095770_RM_Type_IV_5,279114,280784
380020,380020,GCF_008824185_NZ_CP044424_RM_Type_I_1,GCF_008824185_NZ_CP044424_RM_Type_I_1,279124,279124
380021,380021,GCF_008824185_NZ_CP044424_RM_Type_I_1,GCF_023374775_NZ_CP090751_RM_Type_I_1,279124,256701


In [124]:
df_expanded.columns

Index(['Unnamed: 0', 'sys_id', 'Assembly', 'replicon', 'type', 'subtype',
       'sys_beg', 'sys_end', 'protein_in_syst', 'genes_count',
       'name_of_profiles_in_sys', 'accession_in_sys', 'Superkingdom', 'phylum',
       'class', 'order', 'family', 'genus', 'species', 'protein_sequence',
       'cluster_rep_x', 'cluster_rep_y', 'member'],
      dtype='object')

In [125]:
clusters

,index,cluster_rep,member,cluster_idx,member_idx
0,0,GCF_023612275_NZ_CP063766_Abi2_1,GCF_023612275_NZ_CP063766_Abi2_1,0,0
1,1,GCF_023612275_NZ_CP063766_Abi2_1,GCF_001587115_NZ_CP011828_Abi2_1,0,24
2,2,GCF_023612275_NZ_CP063766_Abi2_1,GCF_015377765_NZ_CP045602_Abi2_1,0,32
3,3,GCF_023612275_NZ_CP063766_Abi2_1,GCF_002216705_NZ_CP022340_Abi2_1,0,243
4,4,GCF_023612275_NZ_CP063766_Abi2_1,GCF_003667965_NZ_CP033041_Abi2_1,0,321
...,...,...,...,...,...
380018,380018,GCF_009662315_NZ_CP045739_RM_Type_IV_5,GCF_900243355_NZ_LT969520_RM_Type_IV_2,279114,279285
380019,380019,GCF_009662315_NZ_CP045739_RM_Type_IV_5,GCF_023093935_NZ_CP095770_RM_Type_IV_5,279114,280784
380020,380020,GCF_008824185_NZ_CP044424_RM_Type_I_1,GCF_008824185_NZ_CP044424_RM_Type_I_1,279124,279124
380021,380021,GCF_008824185_NZ_CP044424_RM_Type_I_1,GCF_023374775_NZ_CP090751_RM_Type_I_1,279124,256701


In [81]:
df_expanded = df_expanded.merge(
    clusters,
    left_on="protein_sequence",   # adjust if needed
    right_on="member",
    how="left"
)


In [87]:
df_expanded

,Unnamed: 0,sys_id,Assembly,replicon,type,subtype,sys_beg,sys_end,protein_in_syst,genes_count,...,phylum,class,order,family,genus,species,protein_sequence,cluster_rep_x,cluster_rep_y,member
0,0,GCF_023612275_NZ_CP063766_Abi2_1,GCF_023612275.1,NZ_CP063766,Abi2,Abi2,GCF_023612275.1_NZ_CP063766_01013,GCF_023612275.1_NZ_CP063766_01013,GCF_023612275.1_NZ_CP063766_01013,1.0,...,Bacillota,Bacilli,Lactobacillales,Enterococcaceae,Enterococcus,Enterococcus lactis,MKKYGSKQEYVKRPKSIINNRFPSKLKKKIRPNQYFYSRVRCQIIP...,NaN,NaN,NaN
1,1,GCF_900660665_NZ_LR215037_Abi2_1,GCF_900660665.1,NZ_LR215037,Abi2,Abi2,GCF_900660665.1_NZ_LR215037_00128,GCF_900660665.1_NZ_LR215037_00128,GCF_900660665.1_NZ_LR215037_00128,1.0,...,Mycoplasmatota,Mollicutes,Mycoplasmatales,Mycoplasmataceae,Mycoplasmopsis,Mycoplasmopsis maculosa,MNKNNVKSLLSFEEQLNFLQSKYINWNNQEKFRFKIYLINYGYSHM...,NaN,NaN,NaN
2,2,GCF_900660665_NZ_LR215037_Abi2_2,GCF_900660665.1,NZ_LR215037,Abi2,Abi2,GCF_900660665.1_NZ_LR215037_00677,GCF_900660665.1_NZ_LR215037_00677,GCF_900660665.1_NZ_LR215037_00677,1.0,...,Mycoplasmatota,Mollicutes,Mycoplasmatales,Mycoplasmataceae,Mycoplasmopsis,Mycoplasmopsis maculosa,MNKKIKTILTTDEQINLLEEWGLKWEDDYSRNILKIYLKNYGYKHI...,NaN,NaN,NaN
3,3,GCF_008365255_NZ_CP043501_Abi2_1,GCF_008365255.1,NZ_CP043501,Abi2,Abi2,GCF_008365255.1_NZ_CP043501_01947,GCF_008365255.1_NZ_CP043501_01947,GCF_008365255.1_NZ_CP043501_01947,1.0,...,Bacillota,Bacilli,Bacillales,Bacillaceae,Bacillus,Bacillus paralicheniformis,MVKGKTTNGLMRHLREKHGISINGSKHKKELLNMGYYHGYKGYRFI...,NaN,NaN,NaN
4,4,GCF_009296125_NZ_CP017197_Abi2_1,GCF_009296125.1,NZ_CP017197,Abi2,Abi2,GCF_009296125.1_NZ_CP017197_01353,GCF_009296125.1_NZ_CP017197_01353,GCF_009296125.1_NZ_CP017197_01353,1.0,...,Bacillota,Bacilli,Lactobacillales,Lactobacillaceae,Leuconostoc,Leuconostoc gasicomitatum,MTIYDTEQIKSFLGLRQIKSNDDFKAVETMSTDTAIKFKKYVEYLG...,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
380018,380018,GCF_012029695_NZ_CP061389_Zorya_TypeI_1,GCF_012029695.2,NZ_CP061389,Zorya,Zorya_TypeI,GCF_012029695.2_NZ_CP061389_04302,GCF_012029695.2_NZ_CP061389_04305,GCF_012029695_NZ_CP061389_04305,4.0,...,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Enterobacteriaceae,Klebsiella,Klebsiella pneumoniae,MSWLNSFLVTLTSVEPHNVPFTVISIVSLAVACFLFFYLFRAIKIV...,NaN,NaN,NaN
380019,380019,GCF_002356355_NZ_AP018052_Zorya_TypeI_2,GCF_002356355.1,NZ_AP018052,Zorya,Zorya_TypeI,GCF_002356355.1_NZ_AP018052_01338,GCF_002356355.1_NZ_AP018052_01341,GCF_002356355_NZ_AP018052_01338,4.0,...,Pseudomonadota,Gammaproteobacteria,Chromatiales,NaN,Thiohalobacter,Thiohalobacter thiocyanaticus,MSTKHLAELLDRLANAIVTLEGVALICILIALVALYNYLRFRARIA...,NaN,NaN,NaN
380020,380020,GCF_002356355_NZ_AP018052_Zorya_TypeI_2,GCF_002356355.1,NZ_AP018052,Zorya,Zorya_TypeI,GCF_002356355.1_NZ_AP018052_01338,GCF_002356355.1_NZ_AP018052_01341,GCF_002356355_NZ_AP018052_01339,4.0,...,Pseudomonadota,Gammaproteobacteria,Chromatiales,NaN,Thiohalobacter,Thiohalobacter thiocyanaticus,MKRPAEHTTFEDDGAGYLISVSDMMAGLLFVFMITLMAFVINFQQA...,NaN,NaN,NaN
380021,380021,GCF_002356355_NZ_AP018052_Zorya_TypeI_2,GCF_002356355.1,NZ_AP018052,Zorya,Zorya_TypeI,GCF_002356355.1_NZ_AP018052_01338,GCF_002356355.1_NZ_AP018052_01341,GCF_002356355_NZ_AP018052_01340,4.0,...,Pseudomonadota,Gammaproteobacteria,Chromatiales,NaN,Thiohalobacter,Thiohalobacter thiocyanaticus,MAGLKESLSSRVVTSPHSESWPAPALETAIREARWKLARVEPRDPP...,NaN,NaN,NaN


In [86]:
df_expanded = df_expanded.merge(
    deeptmhmm_df,
    on="cluster_rep",
    how="left"
)

KeyError: 'cluster_rep'

In [82]:
# How many proteins got TM info?
df_expanded["n_tmr"].notna().mean()

# Distribution
df_expanded["n_tmr"].value_counts().sort_index()

# Spot check
df_expanded.loc[
    df_expanded["cluster_rep"] == "0|GCF_023612275_NZ_CP063766_Abi2_1",
    ["sys_id", "n_tmr"]
].head()


KeyError: 'n_tmr'

In [ ]:
df_expanded["is_membrane"] = df_expanded["n_tmr"].fillna(0).gt(0)
